# Plot Styles: Journal-Compliant Figures

geneview ships with a built-in **plot style system** that produces figures compliant to the requirements of major scientific journals — **Nature**, **Science**, and **Cell** — with a single function call.

Each style configures:
- Font family and sizes
- Figure dimensions (column widths)
- Tick mark sizes and directions
- Spine visibility
- Colour palettes (accessible, colour-blind-safe)
- Export settings (DPI, TrueType font embedding)

---

In [ ]:
import matplotlib.pyplot as plt
import geneview as gv
from geneview.plotstyle import apply_style, use_style, list_styles, get_style, PlotStyle, register_style

# Load the built-in GWAS dataset
df = gv.load_dataset("gwas")

# Show available styles
print("Available styles:", list_styles())

## 1. Default geneview Style

The `"geneview"` style is applied automatically when you import geneview. It uses comfortable font sizes (10–12 pt), the legacy geneview colour palette, and hides the top/right spines — suitable for exploration, posters, and general publication.

In [ ]:
apply_style("geneview")

fig, ax = plt.subplots(figsize=(10, 3.5), facecolor="w", edgecolor="k")
ax = gv.manhattanplot(
    data=df[["#CHROM", "POS", "P"]],
    marker=".",
    sign_marker_p=1e-6,
    sign_marker_color="r",
    hline_kws={"linestyle": "--", "lw": 1.0},
    xlabel="Chromosome",
    ylabel=r"$-log_{10}{(P)}$",
    title="Manhattan plot — geneview default style",
    ax=ax,
)
fig.tight_layout()
plt.show()

## 2. Nature Style

The `"nature"` style follows the [Nature Research Figure Guide](https://research-figure-guide.nature.com/figures/preparing-figures-our-specifications):

- Arial / Helvetica, 5–7 pt text
- No background gridlines
- **Wong colour-blind-safe palette** (Nature Methods 8, 441, 2011)
- Single column width ≈ 89 mm (3.5 in)
- 450 dpi raster export

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5), facecolor="w", edgecolor="k")
ax = gv.manhattanplot(
    data=df[["#CHROM", "POS", "P"]],
    marker=".",
    sign_marker_p=1e-6,
    sign_marker_color="r",
    hline_kws={"linestyle": "--", "lw": 1.0},
    xlabel="Chromosome",
    ylabel=r"$-log_{10}{(P)}$",
    title="Manhattan plot — Nature style",
    style="nature",
    ax=ax,
)
fig.tight_layout()
plt.show()

## 3. Science Style

The `"science"` style follows AAAS *Science* guidelines:

- Arial / Helvetica, 6–10 pt text
- **Okabe–Ito** colour-blind-safe palette
- Single column width ≈ 60 mm (2.36 in)
- 600 dpi line-art export

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), facecolor="w", edgecolor="k")
ax = gv.qqplot(
    data=df["P"],
    marker="o",
    title="QQ plot — Science style",
    xlabel=r"Expected $-log_{10}{(P)}$",
    ylabel=r"Observed $-log_{10}{(P)}$",
    style="science",
    ax=ax,
)
fig.tight_layout()
plt.show()

## 4. Cell Style

The `"cell"` style follows Cell Press illustration guidelines:

- Arial / Helvetica, 6–8 pt text
- Accessible colour palette
- Single column width ≈ 85 mm (3.35 in)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), facecolor="w", edgecolor="k")
ax = gv.qqplot(
    data=df["P"],
    marker="o",
    title="QQ plot — Cell style",
    xlabel=r"Expected $-log_{10}{(P)}$",
    ylabel=r"Observed $-log_{10}{(P)}$",
    style="cell",
    ax=ax,
)
fig.tight_layout()
plt.show()

## 5. Side-by-Side Comparison

Below we plot the same Manhattan plot in all four styles for easy visual comparison:

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 14), facecolor="w")

for ax, style_name in zip(axes, ["geneview", "nature", "science", "cell"]):
    gv.manhattanplot(
        data=df[["#CHROM", "POS", "P"]],
        marker=".",
        sign_marker_p=1e-6,
        sign_marker_color="r",
        hline_kws={"linestyle": "--", "lw": 1.0},
        xlabel="Chromosome",
        ylabel=r"$-log_{10}{(P)}$",
        title=f"Manhattan plot — {style_name} style",
        style=style_name,
        ax=ax,
    )

fig.tight_layout(h_pad=3.0)
plt.show()

## 6. Venn Diagrams with Styles

The `style=` parameter also works with `venn()`:

In [ ]:
venn_data = {
    "DEG Study A": {"TP53", "BRCA1", "EGFR", "KRAS", "MYC", "PTEN", "RB1"},
    "DEG Study B": {"BRCA1", "EGFR", "ALK", "BRAF", "MYC", "PIK3CA"},
    "DEG Study C": {"TP53", "KRAS", "BRAF", "NRAS", "IDH1"},
}

fig, axes = plt.subplots(ncols=2, nrows=2, figsize=(12, 12))
for ax, style_name in zip(axes.flat, ["geneview", "nature", "science", "cell"]):
    gv.venn(
        data=venn_data,
        palette="Set2",
        fontsize=11,
        legend_use_petal_color=True,
        style=style_name,
        ax=ax,
    )
    ax.set_title(f"Venn — {style_name} style")

fig.tight_layout()
plt.show()

## 7. Using Styles as a Context Manager

`use_style()` applies a style **temporarily**. When the `with` block exits, the previous style is automatically restored.

In [ ]:
# Default geneview style is active
print("Before context: font size =", plt.rcParams["axes.titlesize"])

with use_style("nature"):
    print("Inside nature: font size =", plt.rcParams["axes.titlesize"])
    fig, ax = plt.subplots(figsize=(10, 3.5), facecolor="w")
    gv.manhattanplot(data=df[["#CHROM", "POS", "P"]], marker=".", ax=ax)
    fig.tight_layout()
    plt.show()

# Back to geneview default
print("After context: font size =", plt.rcParams["axes.titlesize"])

## 8. Inspecting Style Properties

Use `get_style()` to retrieve the full configuration of any registered style:

In [ ]:
for name in list_styles():
    s = get_style(name)
    print(f"\n--- {s.name} ---")
    print(f"  {s.description}")
    print(f"  Font: {s.font_sans_serif[0]}, Title={s.font_size_title}pt, Label={s.font_size_label}pt, Tick={s.font_size_tick}pt")
    print(f"  Figure: {s.figure_figsize[0]:.2f} x {s.figure_figsize[1]:.2f} in, {s.savefig_dpi} dpi")
    print(f"  Palette: {len(s.color_palette)} colours")

## 9. Creating a Custom Style

You can define your own style and register it for use across all geneview plots:

In [ ]:
my_style = PlotStyle(
    name="my_lab",
    description="My lab's house style",
    font_size_title=10.0,
    font_size_label=9.0,
    font_size_tick=7.0,
    figure_figsize=(5.0, 3.5),
    axes_linewidth=0.6,
    color_palette=["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FFC107"],
)
register_style(my_style)

print("Updated styles:", list_styles())

# Use the custom style
fig, ax = plt.subplots(figsize=(10, 3.5), facecolor="w")
ax = gv.manhattanplot(
    data=df[["#CHROM", "POS", "P"]],
    marker=".",
    title="Manhattan plot — my_lab custom style",
    style="my_lab",
    ax=ax,
)
fig.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `apply_style(name)` | Apply a style globally |
| `use_style(name)` | Context manager (temporary) |
| `plot_func(..., style=name)` | Per-call style |
| `list_styles()` | Show available styles |
| `get_style(name)` | Retrieve a `PlotStyle` object |
| `register_style(style)` | Register a custom style |

See also:
- [Plot Styles User Guide](../../docs/user_guide.md#plot-styles)
- [Example script](../../examples/scripts/plotstyle.py)